# Guia Completo: Obtendo Dados da Internet com `requests`
### Referência Prática Organizada em Três Níveis de Complexidade

---
> **Aviso sobre internet:** este tutorial faz requisições reais a servidores públicos. Embora todos os exemplos funcionem **sem chaves (API keys)**, é preciso estar conectado à internet.

## Sumário

| Módulo | Nível | Temas |
|--------|-------|-------|
| **1** | Básico | URLs, Primeira Requisição, Parâmetros, Cabeçalhos |
| **2** | Intermediário | JSON, Erros, Binários, Autenticação, Sessões |
| **3** | Avançado | Webscraping (BeautifulSoup), Tabelas, Ética |

---

## Configuração do Ambiente

Execute a célula abaixo: importa as bibliotecas e **testa a conectividade** antes de começar.

Instalações necessárias no terminal (uma única vez):
```
pip install requests beautifulsoup4 lxml pandas
```

In [ ]:
import json
import requests
import pandas as pd

from urllib.parse import parse_qs, urlencode, urlparse

print('Versão do requests:', requests.__version__)

# Teste rápido de conectividade
try:
    teste = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)
    print('Internet OK — status', teste.status_code)
except requests.RequestException as erro:
    print('SEM internet:', erro)

# MÓDULO 1 — Nível Básico
## URLs, Primeira Requisição, Parâmetros e Cabeçalhos

---
## 1.1 Anatomia de uma URL

Uma **URL** (Uniform Resource Locator) localiza um recurso na internet. Cada parte tem uma função:

| Parte | Exemplo | Função |
|-------|---------|--------|
| **Esquema** | `https` | Protocolo de transporte |
| **Domínio** | `servicodados.ibge.gov.br` | Servidor que responde |
| **Caminho** | `/api/v1/localidades/estados` | Recurso no servidor |
| **Query** | `?orderBy=nome` | Filtros e parâmetros |

`urllib.parse.urlparse()` decompõe uma URL em partes.

In [ ]:
url = 'https://servicodados.ibge.gov.br/api/v1/localidades/estados?orderBy=nome'

partes = urlparse(url)

print('URL completa       :', partes.geturl())
print('| Esquema          :', partes.scheme)
print('| Domínio          :', partes.netloc)
print('| Caminho          :', partes.path)
print('| Query            :', partes.query)
print('Query decodificada :', parse_qs(partes.query))

---
## 1.2 Primeira Requisição GET

`requests.get(url)` envia uma requisição HTTP e retorna um objeto **`Response`**.

**Sintaxe:** `resposta = requests.get(url, timeout=10)`

> **Sempre use `timeout`!** Sem ele, o programa pode ficar travado esperando uma resposta que nunca chega.

In [ ]:
# JSONPlaceholder: API pública de testes (não precisa de chave)
resposta = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)

print('Status HTTP      :', resposta.status_code)
print('Tipo do conteúdo :', resposta.headers.get('content-type'))
print('\n--- Corpo (primeiros 200 caracteres) ---')
print(resposta.text[:200])

---
## 1.3 Entendendo a Resposta

O objeto `Response` guarda tudo que o servidor devolveu: status, cabeçalhos, codificação, tempo e o corpo da resposta.

In [ ]:
resposta = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)

print('Status (200 = sucesso) :', resposta.status_code)
print('URL final              :', resposta.url)
print('Codificação declarada  :', resposta.encoding)
print('Codificação detectada  :', resposta.apparent_encoding)
print('Tempo de resposta (s)  :', round(resposta.elapsed.total_seconds(), 3))

print('\nCabeçalhos de interesse:')
for chave in ['server', 'date', 'content-type', 'content-length']:
    valor = resposta.headers.get(chave)
    print(f'  {chave}: {valor}')

In [ ]:
# Códigos de status HTTP mais comuns
status_comuns = {
    200: 'OK — sucesso',
    201: 'Criado — recurso criado',
    301: 'Redirecionamento permanente',
    400: 'Requisição inválida',
    401: 'Não autenticado',
    403: 'Proibido (sem permissão)',
    404: 'Não encontrado',
    429: 'Muitas requisições (limite excedido)',
    500: 'Erro interno do servidor'
}

for codigo, descricao in status_comuns.items():
    print(f'{codigo}: {descricao}')

---
## 1.4 Parâmetros de Consulta com `params`

Para enviar filtros na URL, use o argumento **`params`** com um dicionário. O `requests` monta a query string corretamente (espaços, acentos, caracteres especiais).

**Sintaxe:** `requests.get(url, params={'chave': 'valor'})`

In [ ]:
filtros = {'userId': 1, '_limit': 3}
resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=filtros,
    timeout=10
)

print('URL montada pelo requests:')
print(resposta.url)
print('\nTotal de posts retornados:', len(resposta.json()))

for post in resposta.json():
    print('-', post['title'])

In [ ]:
# Montando a query string manualmente com urlencode
print('Query:', urlencode({'orderBy': 'nome', 'view': 'nivelado'}))

# API do IBGE: lista de estados do Brasil
resposta = requests.get(
    'https://servicodados.ibge.gov.br/api/v1/localidades/estados',
    params={'orderBy': 'nome'},
    timeout=15
)

estados = resposta.json()
print('Total de estados:', len(estados))
for estado in estados[:5]:
    print(' -', estado['nome'], f"({estado['sigla']})")

---
## 1.5 Cabeçalhos (Headers) Personalizados

Cabeçalhos HTTP enviam informações ao servidor. O **`User-Agent`** identifica a aplicação — vários servidores **recusam** requisições sem um User-Agent reconhecível.

**Sintaxe:** `requests.get(url, headers={'User-Agent': 'MeuApp/1.0'})`

In [ ]:
# httpbingo.org devolve os cabeçalhos que recebeu
resposta = requests.get(
    'https://httpbingo.org/headers',
    headers={'User-Agent': 'AulaDeDados/1.0'},
    timeout=15
)

resposta.json()

In [ ]:
# User-Agent padrão enviado pelo requests
resposta = requests.get('https://httpbingo.org/user-agent', timeout=15)
print('Seu User-Agent padrão:')
resposta.json()

---
## 1.6 O Método `GET` vs `POST`

O `requests` suporta todos os verbos HTTP. Os dois mais usados:

- **`GET`** — pede dados (não muda nada no servidor). Usado para ler/consultar.
- **`POST`** — envia dados ao servidor (normalmente em `json` ou `data`). Usado para criar/alterar.

**Sintaxe:** `requests.get(url)` e `requests.post(url, json={...})`

In [ ]:
# POST enviando dados no formato JSON
novo = {'title': 'Post de teste da aula', 'body': 'Conteúdo', 'userId': 1}

resposta = requests.post(
    'https://jsonplaceholder.typicode.com/posts',
    json=novo,
    timeout=10
)

print('Status:', resposta.status_code, '(201 = criado)')
print('Resposta do servidor:')
print(json.dumps(resposta.json(), indent=2, ensure_ascii=False))

# MÓDULO 2 — Nível Intermediário
## JSON, Erros, Binários, Autenticação e Sessões

---
## 2.1 Trabalhando com Respostas JSON

O método **`.json()`** decodifica automaticamente o corpo da resposta para dicionário/lista Python — pronto para virar um DataFrame.

In [ ]:
resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params={'_limit': 5},
    timeout=10
)

dados = resposta.json()  # lista de dicionários
print('Tipo Python:', type(dados).__name__, '— tamanho:', len(dados))

df_posts = pd.DataFrame(dados)
df_posts[['id', 'userId', 'title']]

In [ ]:
# Exemplo nacional: consulta de CEPs na API pública ViaCEP
ceps = ['01001000', '01310100', '20040002']
registros = []

for cep in ceps:
    resposta = requests.get(f'https://viacep.com.br/ws/{cep}/json/', timeout=10)
    if resposta.ok:
        registros.append(resposta.json())

df_ceps = pd.DataFrame(registros)
df_ceps[['cep', 'logradouro', 'bairro', 'localidade', 'uf']]

---
## 2.2 Tratamento de Erros e Robustez

Nem toda requisição tem sucesso. Use **`raise_for_status()`** (lança exceção quando o status é 4xx/5xx) dentro de um `try`/`except` e defina sempre **`timeout`**.

In [ ]:
def baixar(url, **kwargs):
    """Baixa uma URL de forma segura, retornando Response ou None."""
    try:
        r = requests.get(url, timeout=kwargs.get('timeout', 10))
        r.raise_for_status()
        return r
    except requests.RequestException as erro:
        print(f'Erro ao acessar: {url}')
        print(f'   -> {type(erro).__name__}: {str(erro)[:100]}')
        return None

In [ ]:
# Caso 1: recurso inexistente (404)
r1 = baixar('https://jsonplaceholder.typicode.com/posts/99999')
print('Resultado:', r1)

# Caso 2: CEP inexistente na ViaCEP (retorna 200 com "erro": true)
r2 = baixar('https://viacep.com.br/ws/99999999/json/', timeout=10)
print('Resultado:', r2.json() if r2 else None)

In [ ]:
# Caso 3: timeout — IP de teste que não responde em 3 segundos
try:
    requests.get('https://10.255.255.1/', timeout=3)
except requests.RequestException as erro:
    print(f'Timeout controlado: {type(erro).__name__}')

---
## 2.3 Baixando Conteúdo Binário (Imagens, PDFs, Arquivos)

Em **`resposta.content`** está o dado bruto em bytes — ideal para salvar arquivos. Com **`stream=True`**, o download é feito em partes (`iter_content`), essencial para arquivos grandes.

In [ ]:
# Baixando uma imagem de exemplo (picsum gera uma imagem aleatória)
resposta = requests.get('https://picsum.photos/300/200', timeout=15)
print('Status:', resposta.status_code)
print('Tipo  :', resposta.headers.get('content-type'))
print('Tamanho (bytes):', len(resposta.content))

with open('imagem_exemplo.jpg', 'wb') as arquivo:
    arquivo.write(resposta.content)

print('Imagem salva como imagem_exemplo.jpg')

In [ ]:
# Download em streaming (recomendado para arquivos grandes)
with requests.get('https://picsum.photos/600/400', stream=True, timeout=15) as r:
    r.raise_for_status()
    with open('imagem_stream.jpg', 'wb') as saida:
        for bloco in r.iter_content(chunk_size=1024):
            saida.write(bloco)

print('Download concluído: imagem_stream.jpg')

---
## 2.4 Autenticação

APIs protegidas exigem identificação. As formas mais comuns:

1. **Basic Auth** — usuário e senha direto (`auth=(usuario, senha)`).
2. **Token no cabeçalho** — a API entrega uma chave, enviada como `Authorization: Bearer <token>` (ou `X-Api-Key: <chave>`).

**Sintaxe:** `requests.get(url, auth=(u, s))` ou `requests.get(url, headers={'Authorization': f'Bearer {token}'})`

In [ ]:
# Basic Auth: sem credenciais -> 401; com credenciais -> 200
sem_auth = requests.get('https://httpbingo.org/basic-auth/user/pass', timeout=15)
print('Sem credenciais:', sem_auth.status_code, '(401 = não autenticado)')

com_auth = requests.get(
    'https://httpbingo.org/basic-auth/user/pass',
    auth=('user', 'pass'),
    timeout=15
)
print('Com credenciais:', com_auth.status_code)
com_auth.json()

In [ ]:
# Modelo de envio de token (use o token fornecido pela sua API)
token_exemplo = 'seu_token_aqui'

cabecalhos = {
    'Authorization': f'Bearer {token_exemplo}',
    'Accept': 'application/json'
}

# resposta = requests.get('https://api.exemplo.com/dados', headers=cabecalhos, timeout=10)
print('Cabeçalhos preparados para a requisição:', cabecalhos)

---
## 2.5 Sessões: `requests.Session()`

Uma **`Session`** mantém **cookies**, **cabeçalhos** e a **conexão HTTP** reutilizados entre várias requisições — não é preciso repetir autenticação em cada `get`.

**Sintaxe:** `sessao = requests.Session()` — depois `sessao.get(...)` exatamente como `requests.get(...)`

In [ ]:
sessao = requests.Session()

# O servidor define um cookie na resposta
r1 = sessao.get('https://httpbingo.org/cookies/set?curso=ciencia_de_dados', timeout=15)
print('Cookie definido na 1ª requisição:', r1.json())

# Requisições seguintes da mesma sessão reenviam o cookie automaticamente
r2 = sessao.get('https://httpbingo.org/cookies', timeout=15)
print('Cookie recebido na 2ª requisição:', r2.json())

# Cabeçalho padrão para toda a sessão (vale para os próximos gets)
sessao.headers.update({'User-Agent': 'MinhaSessao/1.0'})
print('Cabeçalho padrão da sessão:', dict(sessao.headers))

# MÓDULO 3 — Nível Avançado
## Webscraping com BeautifulSoup, Tabelas e Ética

---
## 3.1 O que é Webscraping?

**Webscraping** é a técnica de **extrair dados de páginas da web** quando não existe uma API oficial. O fluxo é:

1. o `requests` baixa o **HTML** da página;
2. o **BeautifulSoup** analisa o HTML e localiza os elementos;
3. nós extraímos textos, atributos e links e transformamos em DataFrame/CSV.

Instalação no terminal: `pip install beautifulsoup4 lxml`

### Analisando um HTML simples

In [ ]:
from bs4 import BeautifulSoup

html = '''
<html><body>
  <h1>Minha Loja</h1>
  <p class="produto">Notebook 15"</p>
  <p class="produto destaque">Teclado mecânico</p>
  <p>Mouse sem fio</p>
</body></html>
'''

soup = BeautifulSoup(html, 'html.parser')
print(soup.h1.text)

In [ ]:
# Localizando elementos
titulo = soup.find('h1')
print('Título:', titulo.text)

produtos = soup.find_all('p', class_='produto')
print('Produtos com class="produto":', [p.text for p in produtos])

# Seletores CSS com select()
destaques = soup.select('p.produto.destaque')
print('Destaques:', [p.text for p in destaques])

---
## 3.2 Extração Real: `books.toscrape.com`

O site **books.toscrape.com** foi criado especificamente para estudantes de scraping: página estável com 20 livros por página, títulos, preços e classificações.

> Clicar com o botão direito no navegador → **Inspecionar** ajuda a descobrir os seletores CSS de cada elemento.

In [ ]:
from bs4 import BeautifulSoup

url = 'https://books.toscrape.com/'
resposta = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
resposta.raise_for_status()

soup = BeautifulSoup(resposta.text, 'html.parser')
livros = soup.select('article.product_pod')
print('Livros encontrados na página:', len(livros))

In [ ]:
# Extraindo título, preço, estoque e nota de cada livro
dados = []
for livro in livros[:10]:
    dados.append({
        'titulo': livro.h3.a['title'],
        'preco': livro.select_one('p.price_color').text,
        'estoque': livro.select_one('p.instock.availability').text.strip(),
        'nota': livro.p['class'][1]  # 'One' a 'Five'
    })

df_livros = pd.DataFrame(dados)
df_livros.head()

In [ ]:
# Salvando os dados raspados em CSV
df_livros.to_csv('livros_raspados.csv', index=False, sep=';')
print('Arquivo livros_raspados.csv gerado!')
print(df_livros.to_string())

---
## 3.3 Atalho para Tabelas HTML: `pandas.read_html`

Se o que você precisa já está em uma **tabela (`<table>`)**, o `pd.read_html()` resolve em uma linha — sem BeautifulSoup. Ele detecta todas as tabelas da página e devolve listas de DataFrames.

In [ ]:
import io

# Wikipedia: tabela de população por país
pagina = requests.get(
    'https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)',
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=20
)
print('Página baixada — status', pagina.status_code, '| bytes:', len(pagina.content))

Agora extraímos as tabelas da página:

> **Sintaxe:** `tabelas = pd.read_html(io.StringIO(pagina.text))` — o `StringIO` trata o texto como arquivo em memória.

In [ ]:
tabelas = pd.read_html(io.StringIO(pagina.text))

populacao = tabelas[0]
print('Tabela capturada com shape:', populacao.shape)
populacao.head()

---
## 3.4 Ética, Legalidade e Boas Práticas

Extrair dados da web exige responsabilidade:

1. **Prefira APIs oficiais** — se o site oferece API, use-a (IBGE, ViaCEP, BrasilAPI...).
2. **Respeite o `robots.txt`** — indica o que os robôs podem acessar.
3. **Leia os Termos de Serviço** — raspagens contra as regras podem ser proibidas.
4. **Controle a velocidade** — use `time.sleep()` entre requisições; nunca sobrecarregue o servidor.
5. **Não colete dados pessoais** sem consentimento/autorização.

In [ ]:
# Como verificar o robots.txt de um site (use User-Agent educado)
robots = requests.get(
    'https://en.wikipedia.org/robots.txt',
    headers={'User-Agent': 'Aula de Ciencia de Dados (contato: aluno@escola.edu.br)'},
    timeout=10
)
print(robots.status_code)

# utf-8-sig remove o BOM, evitando problemas de interpretação
print(robots.content.decode('utf-8-sig')[:400])

# Delay educado entre requisições:
# import time
# time.sleep(1.0)

---
## 3.5 Quando o `requests` + `BeautifulSoup` Não Bastam

| Situação | Ferramenta recomendada |
|----------|------------------------|
| Página carregada por **JavaScript** | **Playwright** / **Selenium** |
| Raspagem em **larga escala** (milhares de páginas) | **Scrapy** |
| Necessidade de parsing já embutido | **requests-html** |
| Dados estruturados com API disponível | `requests` (sempre que possível) |

**Lembrete:** antes de automatizar a coleta de qualquer site, confirme se a prática é permitida pelo site e pela legislação (incluindo a LGPD para dados pessoais no Brasil).

---
# Resumo Rápido de Referência

| Função | Descrição |
|--------|-----------|
| `requests.get(url, timeout)` | Requisição GET (ler dados) |
| `requests.post(url, json=...)` | Requisição POST (enviar dados) |
| `requests.Session()` | Sessão com cookies e cabeçalhos reutilizados |
| `.status_code` | Código de status HTTP (200, 404...) |
| `.headers` | Cabeçalhos da resposta (dicionário) |
| `.text` | Corpo da resposta como texto |
| `.content` | Corpo da resposta em bytes (binários) |
| `.json()` | Corpo decodificado como Python |
| `.raise_for_status()` | Lança erro para status 4xx/5xx |
| `params`, `headers`, `auth`, `timeout` | Principais argumentos de `requests` |
| `urlparse(url)` | Decompõe a URL em partes |
| `urlencode({...})` | Monta uma query string |
| `BeautifulSoup(html, 'html.parser')` | Analisa o HTML |
| `.find()`, `.find_all()` | Localiza elementos por tag/classe |
| `.select('css')` | Localiza por seletor CSS |
| `.text`, `['atributo']` | Extrai texto e atributos |
| `pd.read_html(StringIO(texto))` | Extrai tabelas `<table>` de uma página |